In [ ]:
import json
import io
import urllib.request

import pandas as pd

## Load answer key from GitHub

In [2]:
REPO_RAW = 'https://raw.githubusercontent.com/allenai/neurodiscoverybench/main'

TEXT_DATASETS = [
    'SEA-AD-text',
    'WMB-processed-text',
    'WMB-raw-text',
    'WMB-raw-no-traces',
]

In [3]:
# Load answer_key.csv — contains gold hypotheses for text tasks
answer_key_url = f'{REPO_RAW}/eval/answer_key.csv'
with urllib.request.urlopen(answer_key_url, timeout=30) as resp:
    answer_key_text = resp.read().decode()

answer_key_df = pd.read_csv(io.StringIO(answer_key_text))
# Filter to text-only datasets
answer_key_df = answer_key_df[answer_key_df['dataset'].isin(TEXT_DATASETS)].reset_index(drop=True)
print(f'Text-based gold hypotheses: {len(answer_key_df)}')
answer_key_df

Text-based gold hypotheses: 25


,dataset,metadataid,query_id,gold_hypo
0,SEA-AD-text,0,0,"In this dataset, higher ADNC scores are observ..."
1,SEA-AD-text,1,0,"In this dataset, APOE4 allele is observed to b..."
2,SEA-AD-text,2,0,"In this dataset, individuals with high ADNC sc..."
3,SEA-AD-text,3,0,"In this dataset, donors with higher ADNC score..."
4,SEA-AD-text,4,0,"In this dataset, as ADNC scores increase, CERA..."
5,SEA-AD-text,5,0,"In this dataset, donors with high ADNC scores ..."
6,SEA-AD-text,6,0,"In this dataset, higher ADNC scores are observ..."
7,WMB-processed-text,0,0,"In the RHP region, the subclass 019 L2/3 IT PP..."
8,WMB-processed-text,1,0,"In the observed data, class '19 MB Glut' inclu..."
9,WMB-processed-text,2,0,"In the observed data, '24 MY Glut' includes 76..."


## Load questions from metadata JSONs

In [4]:
records = []

for _, row in answer_key_df.iterrows():
    dataset = row['dataset']
    metadata_id = int(row['metadataid'])
    query_id = int(row['query_id'])
    gold_hypo = row['gold_hypo']

    # Fetch the metadata JSON to get the question text
    meta_url = f'{REPO_RAW}/neurodiscoverybench/{dataset}/metadata_{metadata_id}.json'
    try:
        with urllib.request.urlopen(meta_url, timeout=30) as resp:
            metadata = json.loads(resp.read().decode())

        # Extract the question matching query_id
        question = None
        for q in metadata['queries'][0]:
            if q['qid'] == query_id:
                question = q['question']
                break

        if question and pd.notna(gold_hypo):
            # Categorize: SEA-AD questions are about known neuroscience
            # relationships; WMB questions require data analysis
            if 'SEA-AD' in dataset:
                category = 'knowledge'
            else:
                category = 'data-analysis'

            records.append({
                'question': question,
                'answer': gold_hypo,
                'category': category,
                'dataset': dataset,
            })
            print(f'[{dataset}] {question}')
    except Exception as e:
        print(f'Error fetching {meta_url}: {e}')

print(f'\nTotal QA pairs: {len(records)}')

[SEA-AD-text] How does the distribution of Braak stages change across increasing ADNC categories?
[SEA-AD-text] How does the frequency of the APOE4 allele relate with increasing ADNC scores?
[SEA-AD-text] How does the median age at death vary across high and low ADNC categories in the SEA-AD cohort?
[SEA-AD-text] What is the relationship between Thal phases and higher ADNC scores in the SEA-AD cohort?
[SEA-AD-text] How do CERAD scores vary with different ADNC categories in the SEA-AD dataset?
[SEA-AD-text] How does the prevalence of clinically diagnosed dementia differ among donors with varying ADNC scores?
[SEA-AD-text] How does the presence and frequency of comorbid neuropathologies relate with ADNC scores?
[WMB-processed-text] Which subclass of glutamatergic neurons is most frequent in the RHP region?
[WMB-processed-text] What are the number of unique supertypes for the filtered class '19 MB Glut'?
[WMB-processed-text] What are the number of unique supertypes for the filtered class 

## Convert to DataFrame

In [5]:
df = pd.DataFrame(records)
print(f'Total: {len(df)}')
print(f'\nBy category:')
print(df['category'].value_counts())
print(f'\nBy dataset:')
print(df['dataset'].value_counts())
df

Total: 25

By category:
category
data-analysis    18
knowledge         7
Name: count, dtype: int64

By dataset:
dataset
WMB-raw-no-traces     10
SEA-AD-text            7
WMB-processed-text     4
WMB-raw-text           4
Name: count, dtype: int64


,question,answer,category,dataset
0,How does the distribution of Braak stages chan...,"In this dataset, higher ADNC scores are observ...",knowledge,SEA-AD-text
1,How does the frequency of the APOE4 allele rel...,"In this dataset, APOE4 allele is observed to b...",knowledge,SEA-AD-text
2,How does the median age at death vary across h...,"In this dataset, individuals with high ADNC sc...",knowledge,SEA-AD-text
3,What is the relationship between Thal phases a...,"In this dataset, donors with higher ADNC score...",knowledge,SEA-AD-text
4,How do CERAD scores vary with different ADNC c...,"In this dataset, as ADNC scores increase, CERA...",knowledge,SEA-AD-text
5,How does the prevalence of clinically diagnose...,"In this dataset, donors with high ADNC scores ...",knowledge,SEA-AD-text
6,How does the presence and frequency of comorbi...,"In this dataset, higher ADNC scores are observ...",knowledge,SEA-AD-text
7,Which subclass of glutamatergic neurons is mos...,"In the RHP region, the subclass 019 L2/3 IT PP...",data-analysis,WMB-processed-text
8,What are the number of unique supertypes for t...,"In the observed data, class '19 MB Glut' inclu...",data-analysis,WMB-processed-text
9,What are the number of unique supertypes for t...,"In the observed data, '24 MY Glut' includes 76...",data-analysis,WMB-processed-text


## Inspect

In [6]:
print('=== Knowledge questions (RAG-suitable) ===')
for _, row in df[df['category'] == 'knowledge'].iterrows():
    print(f'\nQ: {row["question"]}')
    print(f'A: {row["answer"]}')

print('\n\n=== Data-analysis questions (require raw data) ===')
for _, row in df[df['category'] == 'data-analysis'].iterrows():
    print(f'\nQ: {row["question"]}')
    print(f'A: {row["answer"]}')

=== Knowledge questions (RAG-suitable) ===

Q: How does the distribution of Braak stages change across increasing ADNC categories?
A: In this dataset, higher ADNC scores are observed to be associated with higher Braak stages.

Q: How does the frequency of the APOE4 allele relate with increasing ADNC scores?
A: In this dataset, APOE4 allele is observed to be more frequent in donors with high ADNC compared to those with lower or no ADNC.

Q: How does the median age at death vary across high and low ADNC categories in the SEA-AD cohort?
A: In this dataset, individuals with high ADNC scores are observed to have a higher median age at death compared to those with low ADNC.

Q: What is the relationship between Thal phases and higher ADNC scores in the SEA-AD cohort?
A: In this dataset, donors with higher ADNC scores are observed to exhibit later Thal phases.

Q: How do CERAD scores vary with different ADNC categories in the SEA-AD dataset?
A: In this dataset, as ADNC scores increase, CERAD s

## Save Data

In [7]:
df[['question', 'answer', 'category']].to_csv('neurodiscoverybench.csv', index=False)
print(f'Saved {len(df)} rows to neurodiscoverybench.csv')

Saved 25 rows to neurodiscoverybench.csv
